### Mevcut catalog'lar
Amaç: Free Edition'da hangi catalog'ların olduğunu ve şu an hangi catalog/schema'da çalıştığımızı görmek.
Çıktı: İlk tablo catalog listesi, ikinci tablo aktif catalog ve schema.

In [0]:
%sql
SHOW CATALOGS;

In [0]:
%sql
SELECT current_catalog(), current_schema();

### Medallion schema'ları
Amaç: `workspace` catalog'u altında bronze, silver ve gold schema'larını oluşturmak.
`IF NOT EXISTS`: hücre tekrar çalışırsa hata vermez (idempotent).
Çıktı: Hata yoksa hücre boş sonuç döner; kontrolü sonraki hücrede yapacağız.

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze COMMENT 'Raw Berka data as Delta tables';
CREATE SCHEMA IF NOT EXISTS workspace.silver COMMENT 'Current-state tables, CDC applied with MERGE';
CREATE SCHEMA IF NOT EXISTS workspace.gold   COMMENT 'Dimensional model, SCD Type 2 dim_customer';

### Schema kontrolü
Amaç: `workspace` catalog'undaki schema'ları listelemek.

Çıktı: `bronze`, `silver`, `gold` listede görünmeli. Ayrıca hazır gelen `default` ve `information_schema` da listede olur.

In [0]:
%sql
SHOW SCHEMAS IN workspace;

### Raw volume

Amaç: `workspace.bronze` altında `raw` adında bir managed volume oluşturmak. Berka dosyaları (8 tablo) buraya upload edilecek.

Managed volume: Dosyaların fiziksel yerini Databricks yönetir. Biz sadece `/Volumes/workspace/bronze/raw/` path'ini kullanırız.

Çıktı: Hata yoksa "OK" döner.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.raw COMMENT 'Landing zone for raw Berka files';

### Volume kontrolü

Amaç: `workspace.bronze` altındaki volume'ları listelemek.

Çıktı: `raw` listede görünmeli.

In [0]:
%sql
SHOW VOLUMES IN workspace.bronze;

### Volume path kontrolü

Amaç: Volume'a dosya path'i üzerinden erişebildiğimizi görmek. Berka dosyalarını ileride bu path'ten okuyacağız.

`dbutils.fs.ls()`: Bir klasördeki dosyaları listeler (Databricks'e özel yardımcı fonksiyon).

Çıktı: Volume yeni olduğu için dosya sayısı 0 olmalı. Hata çıkmaması path'in doğru olduğunu gösterir.

In [0]:
raw_volume_path = "/Volumes/workspace/bronze/raw/"

files = dbutils.fs.ls(raw_volume_path)

print("Path:", raw_volume_path)
print("File count:", len(files))

## Landing volume

CDC batch dosyaları için volume. A4'te `src/cdc_generator.py` değişiklikleri buraya JSON lines olarak yazar: `landing/<tablo>/batch_<NNNN>/`.

`IF NOT EXISTS` ile tekrar çalıştırılabilir. Çıktıda hata yoksa volume oluşmuştur.

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS workspace.bronze.landing
COMMENT 'CDC batch files (JSON lines), one folder per table and batch';

### Landing volume kontrolü

`workspace.bronze` altındaki volume'lar listelenir. Çıktıda `raw` ve `landing` olmak üzere iki satır görünmeli.

In [0]:
%sql
SHOW VOLUMES IN workspace.bronze;